In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('ggplot')

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.stats import skew
from sklearn.decomposition import PCA, KernelPCA


In [3]:
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.linear_model import ElasticNet, SGDRegressor, BayesianRidge
from sklearn.kernel_ridge import KernelRidge
#from xgboost import XGBRegressor

In [4]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',1000)

In [5]:
train=pd.read_csv('./X_train.csv')
test=pd.read_csv('./X_test.csv')
price_res=pd.read_csv('./y_train.csv')
train = pd.merge(train, price_res, on="Id") 

In [6]:
train

,Id,鄉鎮市區,交易標的,路名,土地移轉總面積平方公尺,都市土地使用分區,土地數,建物數,車位數,移轉層次,移轉層次項目,總樓層數,建物型態,主要用途,主要建材,建築完成年月,建物移轉總面積平方公尺,建物現況格局-房,建物現況格局-廳,建物現況格局-衛,建物現況格局-隔間,有無管理組織,交易年,交易日,交易月,地鐵站,超商,公園,托兒所,國小,國中,高中職,大學,金融機構,醫院,大賣場,超市,百貨公司,警察局,消防局,縱坐標,橫坐標,單價元平方公尺
0,0,文山區,房地(土地+建物)+車位,興隆路三段,27.75,住,1.0,1.0,1.0,1,無,7.0,華廈(10層含以下有電梯),見其他登記事項,鋼筋混凝土造,2019-10-16,133.43,3,2,2,有,有,2019,31,8,1.0,7.0,2.0,20.0,20.0,19.0,12.0,17.0,15.0,20.0,7.0,20.0,13.0,20.0,16.0,24.957269,121.588026,161672.0
1,1,中正區,房地(土地+建物),金山南路一段,9.57,第三種住宅區,1.0,1.0,0.0,5,無,6.0,華廈(10層含以下有電梯),住家用,鋼筋混凝土造,1997-08-20,40.34,1,1,1,有,有,2021,8,1,1.0,12.0,8.0,20.0,20.0,20.0,13.0,20.0,20.0,20.0,13.0,20.0,20.0,20.0,20.0,24.997141,121.558262,314824.0
2,2,文山區,房地(土地+建物),秀明路一段,9.51,住,1.0,1.0,0.0,1,無,7.0,套房(1房1廳1衛),住家用,鋼筋混凝土造,2009-05-13,70.61,1,1,1,有,有,2020,29,4,0.0,6.0,7.0,20.0,20.0,18.0,11.0,19.0,15.0,18.0,6.0,20.0,11.0,20.0,13.0,24.953906,121.601050,181986.0
3,3,內湖區,房地(土地+建物)+車位,康樂街,23.67,第三種住宅區,1.0,1.0,1.0,10,無,15.0,住宅大樓(11層含以上有電梯),見其他登記事項,鋼筋混凝土造,2009-04-24,143.83,3,2,2,有,有,2019,5,6,1.0,12.0,14.0,20.0,20.0,14.0,5.0,4.0,19.0,20.0,12.0,20.0,13.0,20.0,15.0,25.008046,121.557424,168460.0
4,4,北投區,房地(土地+建物),公路,22.50,第三種住宅區,1.0,1.0,0.0,2,無,5.0,公寓(5樓含以下無電梯),住家用,鋼筋混凝土造,1973-02-15,83.73,2,1,1,有,無,2021,8,4,2.0,4.0,13.0,20.0,20.0,15.0,5.0,15.0,19.0,20.0,10.0,20.0,16.0,18.0,13.0,24.986825,121.557424,134360.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9455,9455,松山區,房地(土地+建物),八德路三段,29.01,第三之一種住宅區,1.0,1.0,0.0,3,無,6.0,華廈(10層含以下有電梯),住家用,鋼筋混凝土造,1975-12-11,130.96,3,2,2,有,有,2021,5,4,2.0,17.0,11.0,20.0,20.0,20.0,14.0,19.0,20.0,20.0,5.0,20.0,20.0,20.0,20.0,25.006934,121.584232,269548.0
9456,9456,文山區,房地(土地+建物),羅斯福路五段,20.48,住,2.0,1.0,0.0,2,無,5.0,公寓(5樓含以下無電梯),住家用,鋼筋混凝土造,1977-12-30,66.24,2,2,2,有,無,2020,15,3,0.0,6.0,9.0,20.0,20.0,20.0,14.0,20.0,15.0,20.0,13.0,20.0,18.0,20.0,20.0,24.968073,121.571759,163043.0
9457,9457,中山區,房地(土地+建物)+車位,基湖路,136.19,住,1.0,1.0,3.0,3,無,9.0,華廈(10層含以下有電梯),住家用,鋼骨鋼筋混凝土造,2001-07-24,433.73,4,2,3,有,有,2020,13,1,1.0,7.0,11.0,20.0,20.0,20.0,6.0,14.0,20.0,20.0,12.0,20.0,19.0,20.0,18.0,25.040072,121.595059,212114.0
9458,9458,南港區,房地(土地+建物)+車位,興南街,39.45,住,1.0,1.0,1.0,11,無,12.0,住宅大樓(11層含以上有電梯),住家用,鋼筋混凝土造,2012-08-13,254.33,0,1,3,有,有,2019,13,11,1.0,11.0,9.0,20.0,20.0,17.0,8.0,6.0,17.0,18.0,12.0,20.0,18.0,20.0,18.0,25.016850,121.630579,185750.0


# data preprocessing

In [14]:
train .columns
for col in train.columns:
    print(col,train[col].dtype
           )

Id int64
鄉鎮市區 object
交易標的 object
路名 object
土地移轉總面積平方公尺 float64
都市土地使用分區 object
土地數 float64
建物數 float64
車位數 float64
移轉層次 int64
移轉層次項目 object
總樓層數 float64
建物型態 object
主要用途 object
主要建材 object
建築完成年月 object
建物移轉總面積平方公尺 float64
建物現況格局-房 int64
建物現況格局-廳 int64
建物現況格局-衛 int64
建物現況格局-隔間 object
有無管理組織 object
交易年 int64
交易日 int64
交易月 int64
地鐵站 float64
超商 float64
公園 float64
托兒所 float64
國小 float64
國中 float64
高中職 float64
大學 float64
金融機構 float64
醫院 float64
大賣場 float64
超市 float64
百貨公司 float64
警察局 float64
消防局 float64
縱坐標 float64
橫坐標 float64
單價元平方公尺 float64


### deal with years

In [ ]:




# Convert '建築完成年月' to datetime format
train['建築完成年月'] = pd.to_datetime(train['建築完成年月'])

# Get the current date
today = datetime.today()

# Calculate the age in 交易年s
train['建築年紀'] = today.year - train['建築完成年月'].dt.year
df = pd.DataFrame()


df['date_str'] = train['交易年'].astype(str) + '-' + train['交易月'].astype(str).str.zfill(2) + '-' + train['交易日'].astype(str).str.zfill(2)
train['交易日期'] = pd.to_datetime(df['date_str'])

print(train['交易日期'])
train['建築年紀']=train['交易日期'].dt.year- train['建築完成年月'].dt.year
print(train['建築年紀'])

0        5
1       27
2       15
3       15
4       51
        ..
9455    49
9456    47
9457    23
9458    12
9459    16
Name: 建築年紀, Length: 9460, dtype: int32


0      2019-08-31
1      2021-01-08
2      2020-04-29
3      2019-06-05
4      2021-04-08
          ...    
9455   2021-04-05
9456   2020-03-15
9457   2020-01-13
9458   2019-11-13
9459   2020-12-28
Name: 交易日期, Length: 9460, dtype: datetime64[ns]
